In [10]:
import pandas as pd
import re


In [11]:

# ── Load data ──────────────────────────────────────────────────────────────────
best = pd.read_csv("../results/best_designs.csv")
best["seq_id"] = [f"seq_{i}" for i in range(len(best))]

foldseek_cols = [
    "query", "target", "fident", "alnlen", "mismatch", "gapopen",
    "qstart", "qend", "tstart", "tend", "evalue", "bits",
]
fs = pd.read_csv(
    "../results/foldseek/foldseek_results.txt",
    sep="\t", header=None, names=foldseek_cols,
)



In [12]:
# Strip _ptm0.XXX suffix to recover seq_id
fs["seq_id"] = fs["query"].str.extract(r"^(seq_\d+)_ptm")

# Pull CATH code from target name when present (af_..._2.60.40.10 style)
fs["cath_hit"] = fs["target"].str.extract(r"(\d+\.\d+\.\d+\.\d+)$")

# Merge foldseek hits with predicted CATH from best_designs
merged = fs.merge(best[["seq_id", "CATHe_Predicted_SFAM", "Sequence"]], on="seq_id", how="left")



In [13]:
fs.head(50)

,query,target,fident,alnlen,mismatch,gapopen,qstart,qend,tstart,tend,evalue,bits,seq_id,cath_hit
0,seq_0_ptm0.361,5fc9B00,0.105,95,83,0,1,94,1,95,0.005918,103,seq_0,NaN
1,seq_0_ptm0.361,1mdaA00,0.144,93,72,0,2,94,19,103,0.008597,95,seq_0,NaN
2,seq_0_ptm0.361,2bzcA00,0.120,86,68,0,17,94,17,102,0.008597,94,seq_0,NaN
3,seq_0_ptm0.361,4hcfA00,0.137,94,74,0,1,94,1,87,0.003829,85,seq_0,NaN
4,seq_0_ptm0.361,af_A8MVW5_252_335_2.60.40.10,0.123,81,68,0,16,94,4,84,0.019300,72,seq_0,2.60.40.10
5,seq_0_ptm0.361,3gdcA01,0.157,87,64,0,18,94,67,153,0.020540,72,seq_0,NaN
6,seq_0_ptm0.361,3zx1A01,0.120,88,67,0,18,94,73,160,0.008597,72,seq_0,NaN
7,seq_0_ptm0.361,3aw5A01,0.141,105,79,0,2,94,11,115,0.018140,68,seq_0,NaN
8,seq_0_ptm0.361,af_O73791_125_215_2.60.40.10,0.085,85,73,0,15,95,6,90,0.049080,67,seq_0,2.60.40.10
9,seq_0_ptm0.361,5oyjD01,0.123,83,69,0,15,94,13,95,0.020540,67,seq_0,NaN


In [14]:
merged.head()

,query,target,fident,alnlen,mismatch,gapopen,qstart,qend,tstart,tend,evalue,bits,seq_id,cath_hit,CATHe_Predicted_SFAM,Sequence
0,seq_0_ptm0.361,5fc9B00,0.105,95,83,0,1,94,1,95,0.005918,103,seq_0,NaN,2.60.40.4100,PSVVEVPKGVLRVFDDLLVTVPANRDLRVIAHQNTEVLTKRLLAAD...
1,seq_0_ptm0.361,1mdaA00,0.144,93,72,0,2,94,19,103,0.008597,95,seq_0,NaN,2.60.40.4100,PSVVEVPKGVLRVFDDLLVTVPANRDLRVIAHQNTEVLTKRLLAAD...
2,seq_0_ptm0.361,2bzcA00,0.120,86,68,0,17,94,17,102,0.008597,94,seq_0,NaN,2.60.40.4100,PSVVEVPKGVLRVFDDLLVTVPANRDLRVIAHQNTEVLTKRLLAAD...
3,seq_0_ptm0.361,4hcfA00,0.137,94,74,0,1,94,1,87,0.003829,85,seq_0,NaN,2.60.40.4100,PSVVEVPKGVLRVFDDLLVTVPANRDLRVIAHQNTEVLTKRLLAAD...
4,seq_0_ptm0.361,af_A8MVW5_252_335_2.60.40.10,0.123,81,68,0,16,94,4,84,0.019300,72,seq_0,2.60.40.10,2.60.40.4100,PSVVEVPKGVLRVFDDLLVTVPANRDLRVIAHQNTEVLTKRLLAAD...


In [15]:
# ── Filter 1: exact CATH code match ───────────────────────────────────────────
# True if ANY foldseek hit for that sequence has the exact predicted SFAM
exact_match = (
    merged.groupby("seq_id")
    .apply(lambda g: (g["cath_hit"] == g["CATHe_Predicted_SFAM"].iloc[0]).any())
    .rename("exact_cath_match")
    .reset_index()
)

# ── Filter 2: 2.60.40.x match (any sub-family) ────────────────────────────────
# True if ANY foldseek hit for that sequence has a CATH code starting with 2.60.40.
broad_match = (
    merged.groupby("seq_id")
    .apply(lambda g: g["cath_hit"].str.startswith("2.60.40.", na=False).any())
    .rename("has_2_60_40_hit")
    .reset_index()
)



/var/folders/fh/pt79pqp11zv19vyr21ngsv480000gp/T/ipykernel_51804/4086174714.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: (g["cath_hit"] == g["CATHe_Predicted_SFAM"].iloc[0]).any())
/var/folders/fh/pt79pqp11zv19vyr21ngsv480000gp/T/ipykernel_51804/4086174714.py:14: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g["cath_hit"].str.startswith("2.60.40.", na=False).any())


In [16]:
# ── Combine and display ────────────────────────────────────────────────────────
result = best.merge(exact_match, on="seq_id").merge(broad_match, on="seq_id")

print(f"Total sequences:        {len(result)}")
print(f"Exact CATH match:       {result['exact_cath_match'].sum()}")
print(f"Has 2.60.40.x hit:      {result['has_2_60_40_hit'].sum()}")

result[[
    "seq_id", "run_id", "CATHe_Predicted_SFAM",
    "exact_cath_match", "has_2_60_40_hit",
]]


Total sequences:        200
Exact CATH match:       8
Has 2.60.40.x hit:      199


,seq_id,run_id,CATHe_Predicted_SFAM,exact_cath_match,has_2_60_40_hit
0,seq_0,20260619_115051_5e7a59,2.60.40.4100,False,True
1,seq_1,20260619_142331_4342aa,2.60.40.1910,False,True
2,seq_2,20260619_130744_44153b,2.60.40.4100,False,True
3,seq_3,20260619_005720_b6e249,2.60.40.4100,False,True
4,seq_4,20260619_065605_5ea6ae,2.60.40.1910,False,True
...,...,...,...,...,...
195,seq_195,20260619_054413_f748aa,2.60.40.1910,False,True
196,seq_196,20260619_080748_7a89c6,2.60.40.4100,False,True
197,seq_197,20260620_135209_41c8ba,2.60.40.1910,False,True
198,seq_198,20260619_115051_5e7a59,2.60.40.3050,False,True


In [18]:
validated = result[result['has_2_60_40_hit']]

In [ ]:
best_validated = validated.head(200)

In [ ]:
best_validated_sequences= best_validated['Sequence']

In [24]:
best_validated_sequences.head()

0    PSVVEVPKGVLRVFDDLLVTVPANRDLRVIAHQNTEVLTKRLLAAD...
1    GTLYLQTEKTVFKQDEVLHLRALIRGEERDWKGVTLRILNDKGELV...
2    QQIVTVSGGVLAHDPASNITIPRNHALQVENKDAVPWVLRHTDMHS...
3    MGFSPASVTLAAGETATLTLVLDDDAVVHGFTLLKDGTPIAPATDG...
4    ENVTVTLDQTSYDIGTTVTVQISLSRAPSSPMIVRLGVLDSHRKLV...
Name: Sequence, dtype: object

In [25]:
best_validated_sequences.to_csv("../results/best_200_validated_sequences.csv", index=False)

In [27]:
with open("../results/best_200_validated_sequences.txt", "w") as f:
    f.write("\n".join(best_validated_sequences))